In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)


# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut/epoch_25.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20
THRESHOLD = 0.5

# =====================================================
# 2. HELPER METRICS (UNCHANGED)
# =====================================================

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

# =====================================================
# 3. DATASET (UNCHANGED)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. MODEL DEFINITION (MC DROPOUT)
# =====================================================
def get_resnet34_binary(pretrained=False, num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_binary(pretrained=False, num_classes=2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model)   # 🔑 keep dropout active

# =====================================================
# 5. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)

        probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            probs_T.append(probs)

        probs_T = torch.stack(probs_T, dim=0)   # [T, B]
        probs_mean = probs_T.mean(dim=0)

        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= THRESHOLD).astype(int)

# =====================================================
# 6. METRICS (UNCHANGED)
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY (UNCHANGED FORMAT)
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")

print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)

# ============================================================
# MODEL EVALUATION — epoch_25.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.8808
# AUC                            | 0.9203
# F1                             | 0.8688
# F2 (β=2)                       | 0.8355
# Macro F1                       | 0.8798
# Macro F2 (β=2)                 | 0.8805
# Precision                      | 0.9307
# Recall                         | 0.8147
# Macro Precision                | 0.8874
# Macro Recall                   | 0.8788
# Specificity                    | 0.9429
# NPV                            | 0.8441
# Cohen Kappa                    | 0.7605
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.054
# Singleton (%) (τ=0.9)          | 94.58
# ECE                            | 0.1023
# ============================================================

# ============================================================
# MODEL EVALUATION — epoch_25.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.8824
# AUC                            | 0.9202
# F1                             | 0.8707
# F2 (β=2)                       | 0.8382
# Macro F1                       | 0.8814
# Macro F2 (β=2)                 | 0.8820
# Precision                      | 0.9309
# Recall                         | 0.8179
# Macro Precision                | 0.8886
# Macro Recall                   | 0.8804
# Specificity                    | 0.9429
# NPV                            | 0.8464
# Cohen Kappa                    | 0.7636
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.060
# Singleton (%) (τ=0.9)          | 93.96
# ECE                            | 0.1053
# ============================================================

# ============================================================
# MODEL EVALUATION — epoch_25.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.8824
# AUC                            | 0.9201
# F1                             | 0.8703
# F2 (β=2)                       | 0.8361
# Macro F1                       | 0.8813
# Macro F2 (β=2)                 | 0.8821
# Precision                      | 0.9341
# Recall                         | 0.8147
# Macro Precision                | 0.8893
# Macro Recall                   | 0.8803
# Specificity                    | 0.9459
# NPV                            | 0.8445
# Cohen Kappa                    | 0.7636
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.057
# Singleton (%) (τ=0.9)          | 94.27
# ECE                            | 0.1044
# ============================================================



--- Loading MC Dropout Model: epoch_25.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference (MC Dropout): 100%|██████████| 90/90 [03:31<00:00,  2.35s/it]


MODEL EVALUATION — epoch_25.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.9361
AUC                            | 0.9719
F1                             | 0.9337
F2 (β=2)                       | 0.9028
Macro F1                       | 0.9360
Macro F2 (β=2)                 | 0.9378
Precision                      | 0.9900
Recall                         | 0.8834
Macro Precision                | 0.9407
Macro Recall                   | 0.9371
Specificity                    | 0.9908
NPV                            | 0.8913
Cohen Kappa                    | 0.8725
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.078
Singleton (%) (τ=0.9)          | 92.18
ECE                            | 0.0391


# NEH on NEH

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
# CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_reduced_split_val_20.csv"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. HELPER METRICS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue

        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. LOAD MODEL
# =====================================================
print(f"\n--- Loading Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 5. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 6. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

# -----------------------
# F2 SCORE (β = 2)
# -----------------------
from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")


print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)

# ============================================================
# MODEL EVALUATION — epoch_8.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.8607
# AUC                            | 0.8876
# F1                             | 0.8427
# Macro F1                       | 0.8588
# Macro Precision                | 0.8722
# Macro Recall                   | 0.8580
# Specificity                    | 0.9459
# NPV                            | 0.8140
# Cohen Kappa                    | 0.7196
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.122
# Singleton (%) (τ=0.9)          | 87.77
# ECE                            | 0.1022
# ============================================================


--- Loading Model: epoch_10.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 16/16 [00:01<00:00,  8.53it/s]


MODEL EVALUATION — epoch_10.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.9577
AUC                            | 0.9947
F1                             | 0.9468
F2 (β=2)                       | 0.9313
Macro F1                       | 0.9559
Macro F2 (β=2)                 | 0.9538
Precision                      | 0.9740
Recall                         | 0.9212
Macro Precision                | 0.9607
Macro Recall                   | 0.9521
Specificity                    | 0.9830
NPV                            | 0.9475
Cohen Kappa                    | 0.9118
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.074
Singleton (%) (τ=0.9)          | 92.56
ECE                            | 0.0245


In [4]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 3. HELPER METRICS FUNCTION
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & LOADER
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 21/21 [00:02<00:00,  8.93it/s]



Applying Venn-Abers Calibration...

EVALUATION: epoch_8.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.8607          | 0.8560         
AUC                            | 0.8876          | 0.8914         
F1                             | 0.8427          | 0.8383         
F2 (β=2)                       | 0.8587          | 0.8543         
Macro F1                       | 0.8588          | 0.8543         
Macro F2 (β=2)                 | 0.8569          | 0.8525         
Precision                      | 0.9305          | 0.9198         
Recall                         | 0.7700          | 0.7700         
Macro Precision                | 0.8722          | 0.8662         
Macro Recall                   | 0.8580          | 0.8535         
Specificity                    | 0.9459          | 0.9369         
NPV                            | 0.8140          | 0.8125  

# NEH on UCSD

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyval_5e4/epoch_10.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. HELPER METRICS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue

        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. LOAD MODEL
# =====================================================
print(f"\n--- Loading Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 5. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 6. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

# -----------------------
# F2 SCORE (β = 2)
# -----------------------

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )
spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")


print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)




--- Loading Model: epoch_10.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 3014/3014 [04:55<00:00, 10.21it/s]


MODEL EVALUATION — epoch_10.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7715
AUC                            | 0.9337
F1                             | 0.8141
F2 (β=2)                       | 0.8902
Macro F1                       | 0.7588
Macro F2 (β=2)                 | 0.7708
Precision                      | 0.7125
Recall                         | 0.9493
Macro Precision                | 0.8115
Macro Recall                   | 0.7613
Specificity                    | 0.5733
NPV                            | 0.9104
Cohen Kappa                    | 0.5327
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.255
Singleton (%) (τ=0.9)          | 74.49
ECE                            | 0.1496


In [3]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 3. HELPER METRICS FUNCTION
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & LOADER
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 3014/3014 [04:30<00:00, 11.12it/s]



Applying Venn-Abers Calibration...

EVALUATION: epoch_8.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.7372          | 0.7183         
AUC                            | 0.9123          | 0.9106         
F1                             | 0.7909          | 0.7810         
F2 (β=2)                       | 0.7242          | 0.7011         
Macro F1                       | 0.7186          | 0.6931         
Macro F2 (β=2)                 | 0.7156          | 0.6911         
Precision                      | 0.6810          | 0.6615         
Recall                         | 0.9430          | 0.9533         
Macro Precision                | 0.7850          | 0.7795         
Macro Recall                   | 0.7254          | 0.7048         
Specificity                    | 0.5078          | 0.4564         
NPV                            | 0.8889          | 0.8976  

# NEH on DHU

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. HELPER METRICS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue

        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. LOAD MODEL
# =====================================================
print(f"\n--- Loading Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 5. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 6. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

# -----------------------
# F2 SCORE (β = 2)
# -----------------------

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")


print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)




--- Loading Model: epoch_8.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 90/90 [01:03<00:00,  1.41it/s]



MODEL EVALUATION — epoch_8.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7714
AUC                            | 0.9621
F1                             | 0.8114
F2 (β=2)                       | 0.8978
Macro F1                       | 0.7606
Macro F2 (β=2)                 | 0.7779
Precision                      | 0.6993
Recall                         | 0.9664
Macro Precision                | 0.8208
Macro Recall                   | 0.7678
Specificity                    | 0.5693
NPV                            | 0.9424
Cohen Kappa                    | 0.5394
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.322
Singleton (%) (τ=0.9)          | 67.82
ECE                            | 0.1311


In [1]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 3. HELPER METRICS FUNCTION
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & LOADER
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 90/90 [00:08<00:00, 10.10it/s]



Applying Venn-Abers Calibration...

EVALUATION: epoch_8.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.7714          | 0.7438         
AUC                            | 0.9621          | 0.9617         
F1                             | 0.8114          | 0.7949         
F2 (β=2)                       | 0.7605          | 0.7282         
Macro F1                       | 0.7606          | 0.7269         
Macro F2 (β=2)                 | 0.7580          | 0.7252         
Precision                      | 0.6993          | 0.6708         
Recall                         | 0.9664          | 0.9753         
Macro Precision                | 0.8208          | 0.8112         
Macro Recall                   | 0.7678          | 0.7396         
Specificity                    | 0.5693          | 0.5039         
NPV                            | 0.9424          | 0.9517  

# NEH on OCTC8

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. HELPER METRICS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue

        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 3. DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. LOAD MODEL
# =====================================================
print(f"\n--- Loading Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 5. INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]

        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)

# =====================================================
# 6. METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

# -----------------------
# F2 SCORE (β = 2)
# -----------------------

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")


print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)




--- Loading Model: epoch_8.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 180/180 [00:23<00:00,  7.57it/s]


MODEL EVALUATION — epoch_8.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7169
AUC                            | 0.8177
F1                             | 0.7561
F2 (β=2)                       | 0.8032
Macro F1                       | 0.7094
Macro F2 (β=2)                 | 0.7141
Precision                      | 0.6888
Recall                         | 0.8380
Macro Precision                | 0.7275
Macro Recall                   | 0.7109
Specificity                    | 0.5837
NPV                            | 0.7662
Cohen Kappa                    | 0.4263
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.354
Singleton (%) (τ=0.9)          | 64.64
ECE                            | 0.1716


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered_imbalanced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 3. HELPER METRICS FUNCTION
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. DATASET & LOADER
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all = [], []
with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference: 100%|██████████| 270/270 [00:24<00:00, 10.87it/s]



Applying Venn-Abers Calibration...

EVALUATION: epoch_8.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.7695          | 0.7714         
AUC                            | 0.8323          | 0.8263         
F1                             | 0.8353          | 0.8404         
F2 (β=2)                       | 0.7678          | 0.7674         
Macro F1                       | 0.7257          | 0.7190         
Macro F2 (β=2)                 | 0.7218          | 0.7115         
Precision                      | 0.8158          | 0.8035         
Recall                         | 0.8556          | 0.8808         
Macro Precision                | 0.7341          | 0.7397         
Macro Recall                   | 0.7197          | 0.7083         
Specificity                    | 0.5837          | 0.5357         
NPV                            | 0.6523          | 0.6759  

: 

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e5"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
BATCH_SIZE = 32
IMG_SIZE = 512
EPOCH_RANGE = (7, 22)   # 7 → 21

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])

        if self.transform:
            img = self.transform(img)

        return img, label


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    model = models.resnet34(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(DEVICE)

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def expected_calibration_error(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# INFERENCE
# =====================================================
print("\n========= UCSD INFERENCE =========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    probs_all, preds_all, labels_all = [], [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]

            probs_all.extend(probs.cpu().numpy())
            preds_all.extend((probs >= 0.5).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    y_true = np.array(labels_all)
    y_pred = np.array(preds_all)
    y_prob = np.array(probs_all)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = expected_calibration_error(y_prob, y_true)
    ent = entropy(y_prob)
    err_unc = ent.mean() * 100

    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ Done.")



========= UCSD INFERENCE =========



/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is e

Epoch 07 | F1=0.8495 | MacroF1=0.8309 | Acc=0.8329 | AUC=0.9369 | Spec=0.7279 | PrecM=0.8480 | RecM=0.8321 | NPV=0.9185 | Kappa=0.6653 | ECE=0.3926 | ErrUnc=23.71% | Set=1 | Singleton=46.66%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 08 | F1=0.8534 | MacroF1=0.8336 | Acc=0.8359 | AUC=0.9418 | Spec=0.7227 | PrecM=0.8538 | RecM=0.8351 | NPV=0.9314 | Kappa=0.6713 | ECE=0.3938 | ErrUnc=23.40% | Set=1 | Singleton=47.85%


...

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 19 | F1=0.8230 | MacroF1=0.7895 | Acc=0.7948 | AUC=0.9296 | Spec=0.6403 | PrecM=0.8252 | RecM=0.7937 | NPV=0.9227 | Kappa=0.5887 | ECE=0.3973 | ErrUnc=22.25% | Set=1 | Singleton=51.22%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 20 | F1=0.8117 | MacroF1=0.7682 | Acc=0.7763 | AUC=0.9308 | Spec=0.5930 | PrecM=0.8181 | RecM=0.7750 | NPV=0.9314 | Kappa=0.5515 | ECE=0.3966 | ErrUnc=22.20% | Set=1 | Singleton=53.56%


Epoch 21 | F1=0.8166 | MacroF1=0.7776 | Acc=0.7845 | AUC=0.9291 | Spec=0.6137 | PrecM=0.8210 | RecM=0.7832 | NPV=0.9274 | Kappa=0.5679 | ECE=0.3957 | ErrUnc=22.50% | Set=1 | Singleton=52.24%

✅ Done.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4"

BATCH_SIZE = 32
IMG_SIZE = 512
EPOCH_RANGE = (7, 22)   # 7 → 21

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])

        if self.transform:
            img = self.transform(img)

        return img, label


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    model = models.resnet34(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(DEVICE)

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def expected_calibration_error(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# INFERENCE
# =====================================================
print("\n========= UCSD INFERENCE =========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    probs_all, preds_all, labels_all = [], [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]

            probs_all.extend(probs.cpu().numpy())
            preds_all.extend((probs >= 0.5).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    y_true = np.array(labels_all)
    y_pred = np.array(preds_all)
    y_prob = np.array(probs_all)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = expected_calibration_error(y_prob, y_true)
    ent = entropy(y_prob)
    err_unc = ent.mean() * 100

    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ Done.")



========= UCSD INFERENCE =========



/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is e

Epoch 07 | F1=0.8301 | MacroF1=0.8100 | Acc=0.8121 | AUC=0.9153 | Spec=0.7118 | PrecM=0.8249 | RecM=0.8114 | NPV=0.8873 | Kappa=0.6237 | ECE=0.4268 | ErrUnc=16.35% | Set=1 | Singleton=49.08%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 08 | F1=0.7839 | MacroF1=0.7238 | Acc=0.7369 | AUC=0.9166 | Spec=0.5232 | PrecM=0.7880 | RecM=0.7353 | NPV=0.9074 | Kappa=0.4721 | ECE=0.4107 | ErrUnc=19.44% | Set=1 | Singleton=56.22%


...

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 19 | F1=0.8350 | MacroF1=0.8071 | Acc=0.8111 | AUC=0.9437 | Spec=0.6712 | PrecM=0.8369 | RecM=0.8101 | NPV=0.9284 | Kappa=0.6214 | ECE=0.4224 | ErrUnc=16.83% | Set=1 | Singleton=53.10%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 20 | F1=0.8161 | MacroF1=0.7749 | Acc=0.7825 | AUC=0.9441 | Spec=0.6040 | PrecM=0.8226 | RecM=0.7811 | NPV=0.9345 | Kappa=0.5638 | ECE=0.4190 | ErrUnc=17.38% | Set=1 | Singleton=55.90%


Epoch 21 | F1=0.8042 | MacroF1=0.7550 | Acc=0.7649 | AUC=0.9412 | Spec=0.5683 | PrecM=0.8119 | RecM=0.7634 | NPV=0.9311 | Kappa=0.5284 | ECE=0.4185 | ErrUnc=17.52% | Set=1 | Singleton=56.97%

✅ Done.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_5e5"

BATCH_SIZE = 32
IMG_SIZE = 512
EPOCH_RANGE = (3, 22)   # 7 → 21

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])

        if self.transform:
            img = self.transform(img)

        return img, label


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    model = models.resnet34(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(DEVICE)

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def expected_calibration_error(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# INFERENCE
# =====================================================
print("\n========= UCSD INFERENCE =========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    probs_all, preds_all, labels_all = [], [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]

            probs_all.extend(probs.cpu().numpy())
            preds_all.extend((probs >= 0.5).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    y_true = np.array(labels_all)
    y_pred = np.array(preds_all)
    y_prob = np.array(probs_all)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = expected_calibration_error(y_prob, y_true)
    ent = entropy(y_prob)
    err_unc = ent.mean() * 100

    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ Done.")



========= UCSD INFERENCE =========



/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is e

Epoch 03 | F1=0.7924 | MacroF1=0.7267 | Acc=0.7425 | AUC=0.9477 | Spec=0.5061 | PrecM=0.8101 | RecM=0.7408 | NPV=0.9530 | Kappa=0.4832 | ECE=0.3686 | ErrUnc=27.36% | Set=1 | Singleton=52.77%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 04 | F1=0.7513 | MacroF1=0.6406 | Acc=0.6747 | AUC=0.9336 | Spec=0.3693 | PrecM=0.7740 | RecM=0.6724 | NPV=0.9371 | Kappa=0.3464 | ECE=0.3794 | ErrUnc=25.17% | Set=1 | Singleton=57.62%


...

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 19 | F1=0.8547 | MacroF1=0.8419 | Acc=0.8429 | AUC=0.9257 | Spec=0.7678 | PrecM=0.8507 | RecM=0.8424 | NPV=0.9011 | Kappa=0.6855 | ECE=0.3920 | ErrUnc=23.79% | Set=1 | Singleton=44.93%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 20 | F1=0.8283 | MacroF1=0.7996 | Acc=0.8037 | AUC=0.9266 | Spec=0.6655 | PrecM=0.8282 | RecM=0.8027 | NPV=0.9160 | Kappa=0.6066 | ECE=0.3821 | ErrUnc=25.01% | Set=1 | Singleton=48.39%


Epoch 21 | F1=0.8044 | MacroF1=0.7607 | Acc=0.7687 | AUC=0.9150 | Spec=0.5905 | PrecM=0.8066 | RecM=0.7674 | NPV=0.9126 | Kappa=0.5362 | ECE=0.3856 | ErrUnc=24.28% | Set=1 | Singleton=50.91%

✅ Done.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_5e4"

BATCH_SIZE = 32
IMG_SIZE = 512
EPOCH_RANGE = (3, 22)   # 7 → 21

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])

        if self.transform:
            img = self.transform(img)

        return img, label


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    model = models.resnet34(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(DEVICE)

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def expected_calibration_error(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# INFERENCE
# =====================================================
print("\n========= UCSD INFERENCE =========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    probs_all, preds_all, labels_all = [], [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]

            probs_all.extend(probs.cpu().numpy())
            preds_all.extend((probs >= 0.5).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    y_true = np.array(labels_all)
    y_pred = np.array(preds_all)
    y_prob = np.array(probs_all)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = expected_calibration_error(y_prob, y_true)
    ent = entropy(y_prob)
    err_unc = ent.mean() * 100

    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ Done.")


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



========= UCSD INFERENCE =========



/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 03 | F1=0.7007 | MacroF1=0.4730 | Acc=0.5714 | AUC=0.8586 | Spec=0.1404 | PrecM=0.7562 | RecM=0.5682 | NPV=0.9719 | Kappa=0.1373 | ECE=0.4581 | ErrUnc=7.79% | Set=1 | Singleton=86.37%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


...

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 19 | F1=0.8859 | MacroF1=0.8814 | Acc=0.8816 | AUC=0.9421 | Spec=0.8502 | PrecM=0.8831 | RecM=0.8814 | NPV=0.9055 | Kappa=0.7631 | ECE=0.4459 | ErrUnc=12.07% | Set=1 | Singleton=46.13%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 20 | F1=0.8796 | MacroF1=0.8725 | Acc=0.8729 | AUC=0.9414 | Spec=0.8235 | PrecM=0.8765 | RecM=0.8725 | NPV=0.9118 | Kappa=0.7456 | ECE=0.4468 | ErrUnc=11.79% | Set=1 | Singleton=47.70%


Epoch 21 | F1=0.8787 | MacroF1=0.8714 | Acc=0.8718 | AUC=0.9409 | Spec=0.8210 | PrecM=0.8757 | RecM=0.8714 | NPV=0.9119 | Kappa=0.7434 | ECE=0.4433 | ErrUnc=12.54% | Set=1 | Singleton=47.53%

✅ Done.
